In [1]:
from julia.api import Julia
from julia import Main

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from multiprocessing import Pool
from concurrent.futures import ProcessPoolExecutor
import concurrent.futures

import scipy.special as sp
import os

import pathlib
from pathlib import Path
import json

#fitters

import pybobyqa
import time
import cma
import csv

def to_float64(df):
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].astype("float64")
    return df

df_predictions = dict([])

Which Fit?

In [2]:
fit_name = "MAP-FD"

Read Files

In [3]:
TMD_fitting_root = "../"
def include(name):
    path = os.path.join(TMD_fitting_root, name)
    Main.eval(f'include(raw"{path}")')

include(f"Cards/{fit_name}.jl")
include(f"DY/DY_table_{Main.flavor_scheme}.jl")

# Data
file_root = f"../Data/{Main.data_name}/Cutted/DY"
matrix_root = f"../Data/{Main.data_name}/Covariance_Matrices/DY"
table_root = f"../Tables/{Main.table_name}/DY"
total_root = f"../Data/DY_total_xsec/{Main.pdf_name}"

initial_params = Main.initial_params

By file or by experiment?

In [4]:
data_selections = "by_experiment"  # "by_file" or "by_experiment"

In [5]:
experiments =[
    'ATLAS_7',
    'ATLAS_8', 
    #'ATLAS_13', 
    'CDF_I',
    'CDF_II',
    'CMS_7',
    'CMS_8',
    'CMS_13',    
    'D0_I',
    'D0_II',
    'D0_II_mu',
    'E288',
    'E605',
    'E772',
    'LHCb_7',
    'LHCb_8',
    'LHCb_13',    
    #'PHENIX',
    'STAR'
]

#["E288","E605","E772","ATLAS", "CMS", "D0", "CDF", "LHCb", "PHENIX", "STAR"]

if data_selections == "by_file":
    file_names = [

    #----------------------------------------------------------------------------
    # ATLAS
    #----------------------------------------------------------------------------

    #"ATLAS/ATLAS_7TeV_y_0_1.csv",
    #"ATLAS/ATLAS_7TeV_y_1_2.csv",
    #"ATLAS/ATLAS_7TeV_y_2_2.4.csv",

    #"ATLAS/ATLAS_8TeV_Q_44_66.csv",
    #"ATLAS/ATLAS_8TeV_Q_116_150.csv",

    #"ATLAS/ATLAS_8TeV_y_0_0.4.csv"
    #"ATLAS/ATLAS_8TeV_y_0.4_0.8.csv"
    #"ATLAS/ATLAS_8TeV_y_0.8_1.2.csv"
    #"ATLAS/ATLAS_8TeV_y_1.2_1.6.csv",
    #"ATLAS/ATLAS_8TeV_y_1.6_2.csv",
    #"ATLAS/ATLAS_8TeV_y_2_2.4.csv",

    #----------------------------------------------------------------------------
    # CDF
    #----------------------------------------------------------------------------

    #"CDF/CDF_RunI.csv",
    #"CDF/CDF_RunII.csv",

    #----------------------------------------------------------------------------
    # CMS
    #----------------------------------------------------------------------------

    #"CMS/CMS_7TeV.csv",
    #"CMS/CMS_8TeV.csv",
    
    #"CMS/CMS_13TeV_y_0_0.4.csv",
    #"CMS/CMS_13TeV_y_0.4_0.8.csv",
    #"CMS/CMS_13TeV_y_0.8_1.2.csv",
    #"CMS/CMS_13TeV_y_1.2_1.6.csv",
    #"CMS/CMS_13TeV_y_1.6_2.4.csv",

    #----------------------------------------------------------------------------
    # D0
    #----------------------------------------------------------------------------

    #"D0/D0_RunI.csv",
    #"D0/D0_RunII.csv",
    #"D0/D0_RunIImu.csv",

    #----------------------------------------------------------------------------
    # LHCb
    #----------------------------------------------------------------------------

    #"LHCb/LHCb_7TeV.csv",
    #"LHCb/LHCb_8TeV.csv",
    #"LHCb/LHCb_13TeV.csv",

    #----------------------------------------------------------------------------
    # Phenix
    #----------------------------------------------------------------------------

    #"PHENIX/PHENIX_200.csv",

    #----------------------------------------------------------------------------
    # STAR
    #----------------------------------------------------------------------------

    #"STAR/STAR_510.csv",

    #----------------------------------------------------------------------------
    # E288
    #----------------------------------------------------------------------------

    #"E288/E288_200_Q_4_5.csv",
    #"E288/E288_200_Q_5_6.csv",
    #"E288/E288_200_Q_6_7.csv",
    #"E288/E288_200_Q_7_8.csv",
    #"E288/E288_200_Q_8_9.csv",
    #"E288/E288_200_Q_10_11.csv",

    #"E288/E288_300_Q_4_5.csv",
    #"E288/E288_300_Q_5_6.csv",
    #"E288/E288_300_Q_6_7.csv",
    #"E288/E288_300_Q_7_8.csv",
    #"E288/E288_300_Q_8_9.csv",
    #"E288/E288_300_Q_10_11.csv",
    #"E288/E288_300_Q_11_12.csv",

    #"E288/E288_400_Q_5_6.csv",
    #"E288/E288_400_Q_6_7.csv",
    #"E288/E288_400_Q_7_8.csv",
    #"E288/E288_400_Q_8_9.csv",
    #"E288/E288_400_Q_10_11.csv",
    #"E288/E288_400_Q_11_12.csv",
    #"E288/E288_400_Q_12_13.csv",
    #"E288/E288_400_Q_13_14.csv",

    #----------------------------------------------------------------------------
    # E605
    #----------------------------------------------------------------------------

    #"E605/E605_Q_7_8.csv",
    #"E605/E605_Q_8_9.csv",
    #"E605/E605_Q_10.5_11.5.csv",
    #"E605/E605_Q_11.5_13.5.csv",
    #"E605/E605_Q_13.5_18.csv",

    #----------------------------------------------------------------------------
    # E772
    #----------------------------------------------------------------------------

    #"E772/E772_Q_5_6.csv",
    #"E772/E772_Q_6_7.csv",
    #"E772/E772_Q_7_8.csv",
    #"E772/E772_Q_8_9.csv",
    #"E772/E772_Q_11_12.csv",
    #"E772/E772_Q_12_13.csv",
    #"E772/E772_Q_13_14.csv",
    #"E772/E772_Q_14_15.csv",
    ]

In [6]:
from pathlib import Path

if data_selections == "by_experiment":
    file_names = []
    for experiment in experiments:
        exp_dir = Path(file_root) / experiment
        for p in sorted(exp_dir.glob("*.csv")):
            file_names.append(str(Path(experiment) / p.name)) 

display(file_names)

['ATLAS_7\\ATLAS7-00y10.csv',
 'ATLAS_7\\ATLAS7-10y20.csv',
 'ATLAS_7\\ATLAS7-20y24.csv',
 'ATLAS_8\\ATLAS8-00y04.csv',
 'ATLAS_8\\ATLAS8-04y08.csv',
 'ATLAS_8\\ATLAS8-08y12.csv',
 'ATLAS_8\\ATLAS8-116Q150.csv',
 'ATLAS_8\\ATLAS8-12y16.csv',
 'ATLAS_8\\ATLAS8-16y20.csv',
 'ATLAS_8\\ATLAS8-20y24.csv',
 'ATLAS_8\\ATLAS8-46Q66.csv',
 'CDF_I\\CDF1.csv',
 'CDF_II\\CDF2.csv',
 'CMS_7\\CMS7.csv',
 'CMS_8\\CMS8.csv',
 'CMS_13\\CMS13-00y04.csv',
 'CMS_13\\CMS13-04y08.csv',
 'CMS_13\\CMS13-08y12.csv',
 'CMS_13\\CMS13-12y16.csv',
 'CMS_13\\CMS13-16y24.csv',
 'D0_I\\D01.csv',
 'D0_II\\D02.csv',
 'D0_II_mu\\D02mu.csv',
 'E288\\E228-200-4Q5.csv',
 'E288\\E228-200-5Q6.csv',
 'E288\\E228-200-6Q7.csv',
 'E288\\E228-200-7Q8.csv',
 'E288\\E228-200-8Q9.csv',
 'E288\\E228-300-11Q12.csv',
 'E288\\E228-300-4Q5.csv',
 'E288\\E228-300-5Q6.csv',
 'E288\\E228-300-6Q7.csv',
 'E288\\E228-300-7Q8.csv',
 'E288\\E228-300-8Q9.csv',
 'E288\\E228-400-11Q12.csv',
 'E288\\E228-400-12Q13.csv',
 'E288\\E228-400-13Q14.csv',


Read Data

In [7]:
data_list = dict()
matrix_list = dict()
df_total_xsec = to_float64(pd.read_csv(f"{total_root}.csv"))
total_xsec_names = df_total_xsec['name'].tolist()

for file in tqdm(file_names):

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    data_list[file] = df_data
    
    matrix = to_float64(pd.read_csv(f"{matrix_root}/{file}"))
    matrix_list[file] = matrix

    name_short = Path(file).stem
    if name_short in total_xsec_names:
        total_xsec = df_total_xsec[df_total_xsec['name'] == name_short]["total_xsec"].values[0]
        data_list[file]['total_xsec'] = total_xsec*np.ones(len(data_list[file]))
        print(f"{name_short}'s total xsec = {total_xsec} added")

  9%|▊         | 5/58 [00:00<00:01, 44.61it/s]

ATLAS7-00y10's total xsec = 264.033 added
ATLAS7-10y20's total xsec = 192.325 added
ATLAS7-20y24's total xsec = 18.4801 added


 31%|███       | 18/58 [00:00<00:01, 34.27it/s]

CMS7's total xsec = 405.098 added
CMS8's total xsec = 483.06 added


 48%|████▊     | 28/58 [00:00<00:00, 36.91it/s]

D02's total xsec = 259.675 added
D02mu's total xsec = 126.199 added


100%|██████████| 58/58 [00:01<00:00, 33.65it/s]


Prediction

In [8]:
Params = Main.Params_Struct(*[np.float32(x) for x in initial_params]) 
#Main.set_params(Main.VRAM, Params) 

for i in range(10):
    Params = Main.Params_Struct(*[np.float32(x) for x in initial_params])                  
    predictions,t = Main.xsec_dict(Main.rel_paths, Main.VRAM)
    print(round(t*1000,2), "ms")

352.58 ms
81.94 ms
82.41 ms
81.84 ms
81.85 ms
82.0 ms
81.86 ms
82.17 ms
82.25 ms
82.6 ms


In [10]:
def get_file_length():

    file_lengths = dict()

    for file in file_names:

        df = to_float64(pd.read_csv(f"{file_root}/{file}"))

        file_lengths[file] = len(df)

    return file_lengths

file_lengths = get_file_length()

def _norm(p: str) -> str:
    return os.path.normpath(p).replace('\\', '/')

def prediction_reformat(predictions):
    preds = {_norm(k): v for k, v in predictions.items()}  # normalize keys once
    df_predictions = {}

    for file in file_names:
        n = file_lengths[file]
        base = os.path.splitext(file)[0]
        xs = []
        for i in range(n):
            table_path = _norm(os.path.join(table_root, f"{base}/{i}.jls"))
            xs.append(preds[table_path])
        df_predictions[file] = np.array(xs)

    return df_predictions

df_predictions = prediction_reformat(predictions)

Chi2

In [11]:
ASWZ_b_array = np.linspace(0.12,0.78,12)*5.067731
ASWZ_prediction = np.array([
    -0.08158508158508182,
    -0.1701631701631705,
    -0.2400932400932403,
    -0.34265734265734293,
    -0.37062937062937085,
    -0.4265734265734267,
    -0.4498834498834501,
    -0.44522144522144536,
    -0.4965034965034967,
    -0.5710955710955714,
    -0.6363636363636365,
    -0.7016317016317017
    ])
ASWZ_upper = np.array([
    0.18414918414918402,
    0.11421911421911402,
    0.09557109557109533,
    0.002331002331002141,
    0.016317016317016098,
    -0.034965034965035224,
    -0.034965034965035224,
    -0.011655011655011815,
    -0.034965034965035224,
    -0.05361305361305391,
    -0.05827505827505863,
    -0.04895104895104918
    ])
ASWZ_error = np.array(ASWZ_upper) - np.array(ASWZ_prediction)

def chi2_lattice(): 
    CS_list = []
    for b in ASWZ_b_array :
        Q = 2.0
        CS = Main.CS_total_func(b, Q)
        CS_list.append(CS)
    chi2dN = np.sum( (CS_list - ASWZ_prediction)**2 / ASWZ_error**2 ) / len(ASWZ_b_array)
    return chi2dN

def timed(func):
    t0 = time.perf_counter()
    out = func()
    return out, time.perf_counter() - t0

#chi2dN, t = timed(chi2_lattice)
#print("χ^2/N from LATTICE =", chi2dN, ", took", round(t, 4), "seconds")

In [12]:
def get_chi2dN(df_predictions):

    N_list = dict()
    chi2dN_list = dict()
    chi2_total = 0.0
    N_total = 0

    for file in file_names:

        data_xsec = data_list[file]["xsec"].to_numpy()
        pred_xsec = df_predictions[file]
        diff_xsec = data_xsec - pred_xsec

        covariance_matrix_inv = matrix_list[file].to_numpy()

        N = len(data_xsec)

        chi2 = diff_xsec @ covariance_matrix_inv @ diff_xsec

        chi2_total += chi2
        N_total += N
        chi2dN_list[file] = float(round(chi2/N, 2))
        N_list[file] = N

    chi2dN = chi2_total / N_total
    return chi2dN, chi2dN_list, N_list

chi2dN, chi2dN_list, N_list = get_chi2dN(df_predictions)

print(f"Total χ^2/N = {chi2dN:.2f}")
display(chi2dN_list)

Total χ^2/N = 1.56


{'ATLAS_7\\ATLAS7-00y10.csv': 6.89,
 'ATLAS_7\\ATLAS7-10y20.csv': 10.25,
 'ATLAS_7\\ATLAS7-20y24.csv': 4.44,
 'ATLAS_8\\ATLAS8-00y04.csv': 5.8,
 'ATLAS_8\\ATLAS8-04y08.csv': 4.73,
 'ATLAS_8\\ATLAS8-08y12.csv': 4.0,
 'ATLAS_8\\ATLAS8-116Q150.csv': 0.61,
 'ATLAS_8\\ATLAS8-12y16.csv': 6.72,
 'ATLAS_8\\ATLAS8-16y20.csv': 4.72,
 'ATLAS_8\\ATLAS8-20y24.csv': 2.35,
 'ATLAS_8\\ATLAS8-46Q66.csv': 2.09,
 'CDF_I\\CDF1.csv': 0.59,
 'CDF_II\\CDF2.csv': 1.39,
 'CMS_7\\CMS7.csv': 1.59,
 'CMS_8\\CMS8.csv': 1.59,
 'CMS_13\\CMS13-00y04.csv': 2.13,
 'CMS_13\\CMS13-04y08.csv': 1.41,
 'CMS_13\\CMS13-08y12.csv': 0.36,
 'CMS_13\\CMS13-12y16.csv': 0.21,
 'CMS_13\\CMS13-16y24.csv': 0.25,
 'D0_I\\D01.csv': 0.53,
 'D0_II\\D02.csv': 0.5,
 'D0_II_mu\\D02mu.csv': 0.22,
 'E288\\E228-200-4Q5.csv': 0.25,
 'E288\\E228-200-5Q6.csv': 0.78,
 'E288\\E228-200-6Q7.csv': 0.92,
 'E288\\E228-200-7Q8.csv': 0.99,
 'E288\\E228-200-8Q9.csv': 0.68,
 'E288\\E228-300-11Q12.csv': 0.24,
 'E288\\E228-300-4Q5.csv': 0.83,
 'E288\\E228-300-

Objective

In [13]:
def objective(params):
    try:
        params_cl = Main.Params_Struct(*[np.float32(x) for x in params])
        Main.set_params(Main.VRAM, params_cl)

        predictions, t = Main.xsec_dict(Main.rel_paths, Main.VRAM)

        df_predictions = prediction_reformat(predictions)
        chi2dN, _, _ = get_chi2dN(df_predictions)

        if not np.isfinite(chi2dN): 
            return 1e5
        return chi2dN

    except Exception as e:
        return 1e5

objective(initial_params)

np.float64(1.5570237322838556)

Random

In [14]:
rng = np.random.default_rng()
def generate_gaussian_data():

    data_list = dict()

    for file in file_names:
        df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
        xsecs = df_data["xsec"].to_numpy(np.float64)

        cov_inv = np.asarray(matrix_list[file], np.float64)
        cov_inv = 0.5 * (cov_inv + cov_inv.T)
        cov = np.linalg.inv(cov_inv)
        cov = 0.5 * (cov + cov.T)

        eps = np.finfo(cov.dtype).eps * np.max(np.diag(cov))
        L = np.linalg.cholesky(cov + eps * np.eye(len(xsecs)))

        df_data["xsec"] = xsecs + L @ rng.standard_normal(len(xsecs))
        data_list[file] = df_data

    return data_list

In [15]:
data_list = generate_gaussian_data()
for file in file_names:

    df_data = to_float64(pd.read_csv(f"{file_root}/{file}"))
    xsecs = df_data["xsec"].to_numpy()

    cov_mat_inv = matrix_list[file]
    cov_mat = np.linalg.inv(cov_mat_inv)
    cov_diag = np.diag(cov_mat)
    errors = np.sqrt(cov_diag)

    diff = (data_list[file]["xsec"].to_numpy() - xsecs)/errors
    print(f"diff: {diff}")

diff: [-0.7489084  -0.6221179   0.10786162  0.15806502  0.17255768 -0.01803445]
diff: [-0.7856522   0.41751987 -0.0573388   0.22616544 -0.40150423 -0.13187059]
diff: [ 1.33213177  0.2295668   0.16278892  1.89443474 -0.27987926 -0.71514478]
diff: [-0.86748607 -0.8947363  -0.78935534 -0.85518352 -0.96370713 -0.80900904]
diff: [0.38278416 0.30828472 0.38167043 0.28609122 0.02461736 0.33937409]
diff: [-0.19069016 -0.20331862 -0.18230952 -0.00575257 -0.04625626 -0.33854334]
diff: [-0.03893354  0.32196857 -0.11106901  0.37078891 -0.02494158  0.52423028
  0.64648099  0.70584451]
diff: [0.84988213 0.79123642 0.84267542 0.99066513 0.81965802 1.16411647]
diff: [ 0.22621693  0.10126481  0.1856024   0.24140182  0.13170558 -0.07462447]
diff: [0.25613469 0.34741597 0.5504697  0.47503192 0.80205507 0.44387153]
diff: [-0.32170113  1.0008024   0.53946765  0.97735661]
diff: [ 0.85901104 -0.4876598  -0.89400837  0.98070472 -0.89307944 -1.06901581
 -2.20970163 -1.20556787  0.0445798  -1.11120511  0.127624

In [16]:
objective(initial_params)

np.float64(2.331467633848725)

Bounds

In [18]:
bounds_raw = Main.bounds_raw
lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

In [19]:
def objective_normalized(params):

    normalized_params = lower_bounds + params * (upper_bounds - lower_bounds)

    return objective(normalized_params)

def objective_log(params):
    return np.log10(objective_normalized(params))

def normalize_params(params):
    return (params - lower_bounds) / (upper_bounds - lower_bounds)

def denormalize_params(params):
    return lower_bounds + params * (upper_bounds - lower_bounds)

theta0 = normalize_params(np.array(initial_params))
dim = len(bounds_raw)

In [20]:
lower_bounds, upper_bounds = np.array(list(zip(*bounds_raw)))

theta0 = normalize_params(np.array(initial_params))
dim = len(bounds_raw)

print(f"initial chi2: {round(objective_normalized(theta0), 3)}")  

res = pybobyqa.solve(
        objective_log, theta0,
        bounds=(np.zeros(dim), np.ones(dim)),
        maxfun=10000,
        rhobeg=0.1,
        rhoend=1e-6,
        scaling_within_bounds=True,
        seek_global_minimum=True, 
    )

initial chi2: 2.331


In [ ]:
best_chi2dN = 10**(res.f)
best_chi2dN

np.float64(2.2840961323621642)

Global Minimize

Py-Bobyqa Fit

In [ ]:
N_replicas = 200

for i in tqdm(range(N_replicas)):

    data_list = generate_gaussian_data()

    res = pybobyqa.solve(
            objective_log, theta0,
            bounds=(np.zeros(dim), np.ones(dim)),
            maxfun=10000,
            rhobeg=0.1,
            rhoend=1e-6,
            scaling_within_bounds=True,
            seek_global_minimum=True, 
        )
    
    best_chi2dN = 10**(res.f)
    optimal_params_normalized = res.x
    optimal_params = denormalize_params(optimal_params_normalized)

    fmt = lambda x: f"{x:.5g}"     
    with open("replicas.csv", "a", newline="") as f:
        csv.writer(f).writerow([i, fmt(best_chi2dN), *map(fmt, np.asarray(optimal_params))])

    print(f"best chi2: {fmt(best_chi2dN)}")